# Neural Networks From Scratch

In [3]:
import numpy as np
import math
import random

In [4]:
# activation functions

def sigmoid(x):
    return (1/(1+np.exp(-x)))

def tanh(x):
    return (np.exp(x)-np.exp(-x))/(np.exp(x)+np.exp(-x))

def relu(x):
    return x if x > 0 else 0


In [ ]:
class Layer:
    def __init__(self):
        # Base class for layers in the network.
        # Subclasses should implement forward/backward and optionally update.
        pass

    def forward(self, x):
        raise NotImplementedError("forward must be implemented by Layer subclasses")

    def backward(self, d_out):
        raise NotImplementedError("backward must be implemented by Layer subclasses")

    def update(self, lr):
        # Default no-op update for layers without learnable parameters.
        return None


class ReLU(Layer):
    def forward(self, x):
        self.mask = x > 0
        return x * self.mask

    def backward(self, d_out):
        return d_out * self.mask

    def update(self, lr):
        # ReLU has no parameters to update.
        return None

class MSELoss:
    def forward(self, y_pred, y):
        self.y_pred = y_pred
        self.y = y
        return np.mean((y_pred - y)**2)

    def backward(self):
        return 2 * (self.y_pred - self.y) / self.y.size

### Network = (just a pipeline)

The network does not do math itself.
It just chains layers:

$X→L1→L2→L3→\hat{y}$

In [6]:
class Dense(Layer):
    def __init__(self, inp_dim, out_dim):
        self.W = np.random.randn(out_dim, inp_dim) * 0.01
        self.b = np.zeros((out_dim,1))

    def forward(self, x):
        assert x.shape[0] == self.W.shape[1], f"Expected {self.W[1].shape}, Got {x.shape[0]}"

        self.x = x
        return self.W @ x + self.b

    def backward(self, d_out):
        N = self.x.shape[1]

        self.dW = (d_out @ self.x.T) / N     # (out_dim, in_dim)
        self.db = np.sum(d_out, axis=1, keepdims=True) / N  # (out_dim, 1)

        dx = self.W.T @ d_out  # (in_dim, N)
        return dx

    def update(self, lr):
        self.W -= lr * self.dW
        self.b -= lr * self.db


class NeuralNetwork:
    def __init__(self, layers):
        self.layers = layers

    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)
        return x

    def backward(self, grad):
        for layer in reversed(self.layers):
            grad = layer.backward(grad)

    def update(self, lr):
        for layer in self.layers:
            layer.update(lr)
            

In [7]:
from sklearn.datasets import make_moons

### Intuition behind Learning Rate

Learning rate controls the step size in parameter space

Too high (e.g. 1.0)
* loss oscillates or explodes
* training unstable

Too low (e.g. 1e-5)
* loss barely decreases
* training extremely slow

In [8]:
X, y = make_moons(n_samples=500, noise=0.1)

In [9]:
layers = [2, 16, 8, 1]

network_layers = []
for i in range(len(layers)-1):
    network_layers.append(Dense(layers[i], layers[i+1]))
    if (i < len(layers) - 2):
        network_layers.append(ReLU())

network = NeuralNetwork(network_layers)

loss_func = MSELoss()
epochs = 2000 # 1 epoch means one full pass over the data 
# more epochs = more refinement but diminishing returns after some point
lr = 0.01

for epoch in range(epochs):
    y_pred = network.forward(X)
    loss = loss_func.forward(y_pred, y)
    grad = loss_func.backward()

    if epoch % 100 == 0:
        print(epoch, loss)

    network.backward(grad)
    network.update(lr)

AssertionError: Expected (2,), Got 500